# Notebook 3 — Phase 1: L-BFGS (adaptive) | Phase 2: Adam (adaptive) | Phase 3: NGD

| Phase | Space | Optimizer | LR strategy |
|-------|-------|-----------|-------------|
| 1 | W   | L-BFGS | adaptive: `global_lr_scale × 0.08 × LBFGS_SCALE[level]` |
| 2 | W+  | Adam   | adaptive: `global_lr_scale × 0.02 × ADAM_SCALE[level]` |
| 3 | W++ | NGD    | fixed: `global_lr_scale × 0.005` |

This mirrors the original run.py combo (LBFGS → Adam → NGD) but now with
level-aware adaptive learning rates for L-BFGS and Adam.

Run simultaneously with Notebook 1 and Notebook 2 on separate Kaggle sessions.

In [ ]:
# ── 0. Kaggle environment setup ───────────────────────────────────────────────
import os, sys, subprocess

REPO_DIR = "/kaggle/working/stylegan"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/BayesianDecoder/stylegan.git", REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)

# ── Register every directory that contains importable packages ────────────────
# 1. stylegan2_ada/ owns dnnlib, torch_utils, training — MUST be first
STYLEGAN2_DIR = os.path.join(REPO_DIR, "stylegan2_ada")
sys.path.insert(0, STYLEGAN2_DIR)
sys.path.insert(0, REPO_DIR)

# 2. robust_unsupervised may be nested inside robust-unsupervised/ on GitHub
#    (GitHub repo layout) vs directly under REPO_DIR (local layout)
for candidate in [
    os.path.join(REPO_DIR, "robust-unsupervised"),   # GitHub nested layout
    REPO_DIR,                                          # flat local layout
]:
    if os.path.isdir(os.path.join(candidate, "robust_unsupervised")):
        if candidate not in sys.path:
            sys.path.insert(0, candidate)
        print(f"  robust_unsupervised → {candidate}")
        break

# ── Install Python deps ───────────────────────────────────────────────────────
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "tyro", "torchmetrics", "lpips", "pytorch-fid", "ninja"],
    check=True,
)

# ── Sanity-check every critical module before proceeding ─────────────────────
import importlib.util
all_ok = True
for mod in ["dnnlib", "torch_utils", "robust_unsupervised", "benchmark"]:
    found = importlib.util.find_spec(mod) is not None
    print(f"  {mod:25s}: {'OK' if found else 'NOT FOUND'}")
    if not found:
        all_ok = False

if not all_ok:
    print("\n[DEBUG] sys.path =", "\n  ".join(sys.path))
    raise RuntimeError("Missing modules — see NOT FOUND lines above and fix paths.")

print("\nSetup complete")

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import glob, datetime
import torch
import tqdm
import torchmetrics
import torchmetrics.image.lpip as lpips_metrics
import torchvision.transforms as TVT
from torchvision.io import read_image, write_png

import benchmark
from robust_unsupervised import (
    NGD, LBFGSPhase,
    WVariable, WpVariable, WppVariable,
    MultiscaleLPIPS, open_generator, open_image,
    save_image, resize_for_logging, directory,
)
print("Imports OK")

In [ ]:
# ── 2. Config — UPDATE THESE PATHS to match your Kaggle dataset names ─────────
#
# Upload these 3 items as separate Kaggle Datasets before running:
#
#   1. ffhq.pkl              → dataset name e.g. "stylegan-ffhq"
#                              mounts at /kaggle/input/stylegan-ffhq/
#
#   2. FFHQ-X images folder  → dataset name e.g. "ffhq-x-dataset"
#                              mounts at /kaggle/input/ffhq-x-dataset/
#
#   3. FFHQ-X_crops128_ncrops1000.npz (pFID reference, 31 MB)
#                            → dataset name e.g. "ffhq-pfid-ref"
#                              mounts at /kaggle/input/ffhq-pfid-ref/
#
CFG = dict(
    name            = "nb3_LBFGS_Adam_NGD",
    pkl_path        = "/kaggle/input/stylegan-ffhq/ffhq.pkl",
    dataset_path    = "/kaggle/input/ffhq-x-dataset/FFHQ-X",
    pfid_ref        = "/kaggle/input/ffhq-pfid-ref/FFHQ-X_crops128_ncrops1000.npz",
    resolution      = 1024,
    global_lr_scale = 1.0,
    tasks           = "all",   # "all" | "single" | "composed"
    task_name       = None,
    levels          = None,
    n_images        = None,
    steps           = 150,
)

benchmark.config.resolution = CFG["resolution"]
timestamp = datetime.datetime.now().isoformat(timespec="seconds").replace(":", "")
print(f"Run tag: {CFG['name']}/{timestamp}")

In [ ]:
# ── 3. Adaptive LR tables ─────────────────────────────────────────────────────
#
# actual_lr = global_lr_scale × base_phase_lr × SCALE[level]
#
# L-BFGS (Phase 1, W space):
#   Uses quasi-Newton curvature.  At XL the loss surface is rough so the
#   curvature estimate becomes unreliable — small scale (0.1) prevents the
#   strong-Wolfe line-search from diverging.  At XS the surface is smooth
#   and scale=1.0 lets the solver converge in fewer steps.
#
# Adam (Phase 2, W+ space):
#   Adam's second-moment estimate already adapts to gradient variance, so
#   the reduction across levels is gentler than for L-BFGS.  At XL
#   (scale=0.3) we still reduce to prevent overshooting in high-curvature
#   directions that Adam's momentum term might amplify.
#
# NGD (Phase 3, W++ space):
#   Gradient direction is normalised, so the step magnitude is already
#   scale-invariant — a single base LR works for all levels.

LBFGS_SCALE = {
    "XS": 1.0,
    "S" : 0.8,
    "M" : 0.5,
    "L" : 0.3,
    "XL": 0.1,
}

ADAM_SCALE = {
    "XS": 1.0,
    "S" : 0.9,
    "M" : 0.7,
    "L" : 0.5,
    "XL": 0.3,
}

# Composed tasks use integer levels → map to string keys
_INT_TO_LEVEL = {2: "S", 3: "M", 4: "L"}

def _level_key(level) -> str:
    if isinstance(level, int):
        return _INT_TO_LEVEL.get(level, "M")
    return str(level)

_BASE_LR = {"W": 0.08, "W+": 0.02, "W++": 0.005}

def phase_lr(phase: str, optimizer_name: str, level, gls: float) -> float:
    base = _BASE_LR[phase]
    lk   = _level_key(level)
    if optimizer_name == "LBFGS":
        scale = LBFGS_SCALE.get(lk, 0.5)
    elif optimizer_name == "Adam":
        scale = ADAM_SCALE.get(lk, 0.7)
    else:   # NGD
        scale = 1.0
    return gls * base * scale

print("LR tables:")
for lvl in ["XS", "S", "M", "L", "XL"]:
    w   = phase_lr("W",   "LBFGS", lvl, 1.0)
    wp  = phase_lr("W+",  "Adam",  lvl, 1.0)
    wpp = phase_lr("W++", "NGD",   lvl, 1.0)
    print(f"  {lvl:2s} | W(L-BFGS)={w:.5f}  W+(Adam)={wp:.5f}  W++(NGD)={wpp:.6f}")

In [ ]:
# ── 4. Scoring & pFID helpers ─────────────────────────────────────────────────
_SCORE_DEV = "cuda" if torch.cuda.is_available() else "cpu"
_psnr      = torchmetrics.PeakSignalNoiseRatio(data_range=2.0).to(_SCORE_DEV)
_lpips_m   = lpips_metrics.LearnedPerceptualImagePatchSimilarity(net_type="vgg").to(_SCORE_DEV)

_CROP_RES  = 128
_CROP_NUM  = 1000
_FID_REF   = CFG["pfid_ref"]   # points to Kaggle input dataset (.npz)


def compute_scores(pred, gt):
    with torch.no_grad():
        p = (pred * 2.0 - 1.0).clamp(-1, 1).to(_SCORE_DEV)
        g = (gt   * 2.0 - 1.0).clamp(-1, 1).to(_SCORE_DEV)
        return _psnr(p, g).item(), _lpips_m(p, g).item()


def make_pfid_crops(image_paths, out_dir):
    import shutil
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    crop   = TVT.RandomCrop(_CROP_RES)
    n_each = max(1, _CROP_NUM // len(image_paths))
    idx    = 0
    for path in image_paths:
        img = read_image(path)
        if img.shape[1] < _CROP_RES or img.shape[2] < _CROP_RES:
            continue
        for _ in range(n_each):
            write_png(crop(img), os.path.join(out_dir, f"{idx:05d}.png"))
            idx += 1


def compute_pfid(pred_paths, work_dir, suffix="_W++"):
    if not os.path.exists(_FID_REF):
        print(f"  [pFID] reference not found: {_FID_REF}"); return None
    crops_dir = os.path.join(work_dir, f"pfid_crops{suffix.replace('/', '_')}")
    print(f"  Making {_CROP_NUM} crops for pFID...")
    make_pfid_crops(pred_paths, crops_dir)
    try:
        out = subprocess.check_output(
            ["python", "-m", "pytorch_fid", _FID_REF, crops_dir],
            cwd=REPO_DIR, stderr=subprocess.STDOUT,
        )
        return float(out.decode().strip().splitlines()[-1].replace("FID:  ", "").strip())
    except Exception as e:
        print(f"  [pFID] failed: {e}"); return None


def _mean(vals):
    return sum(vals) / len(vals) if vals else float("nan")

print("Helpers defined")

In [ ]:
# ── 5. Phase runner ───────────────────────────────────────────────────────────
def run_phase(label, variable, optimizer_cls, lr, degradation, target,
              ground_truth, steps, resolution):
    """Optimise `variable` for `steps` steps; return {PSNR, LPIPS}."""
    optimizer = optimizer_cls(variable.parameters(), lr=lr)

    try:
        for _ in tqdm.tqdm(range(steps), desc=label, leave=False):
            def closure():
                optimizer.zero_grad()
                x    = variable.to_image()
                loss = loss_fn(
                    degradation.degrade_prediction, x, target, degradation.mask
                ).mean()
                loss.backward()
                return loss

            if isinstance(optimizer, torch.optim.LBFGS):
                optimizer.step(closure)
            else:
                closure()
                optimizer.step()
    except KeyboardInterrupt:
        pass

    suffix   = "_" + label
    pred     = resize_for_logging(variable.to_image(), resolution)
    deg_pred = degradation.degrade_ground_truth(pred)
    approx   = degradation.degrade_prediction(pred)

    save_image(pred,     f"pred{suffix}.png",          padding=0)
    save_image(deg_pred, f"degraded_pred{suffix}.png", padding=0)
    save_image(torch.cat([approx, deg_pred]),
               f"degradation_approximation{suffix}.jpg", padding=0)
    save_image(
        torch.cat([
            ground_truth,
            resize_for_logging(target, resolution),
            resize_for_logging(deg_pred, resolution),
            pred,
        ]),
        f"side_by_side{suffix}.jpg", padding=0,
    )
    save_image(torch.cat([resize_for_logging(target, resolution), pred]),
               f"result{suffix}.jpg", padding=0)
    save_image(
        torch.cat([ground_truth, pred, (ground_truth - pred).abs()]),
        f"accuracy{suffix}.jpg", padding=0,
    )

    psnr, lpips = compute_scores(pred, ground_truth)
    return {"PSNR": psnr, "LPIPS": lpips}

print("run_phase defined")

In [ ]:
# ── 6. Load generator & loss ──────────────────────────────────────────────────
G       = open_generator(CFG["pkl_path"])
loss_fn = MultiscaleLPIPS()
print("Generator loaded")

In [ ]:
# ── 7. Task selection ─────────────────────────────────────────────────────────
if   CFG["tasks"] == "single":   tasks = benchmark.single_tasks
elif CFG["tasks"] == "composed": tasks = benchmark.composed_tasks
else:                             tasks = benchmark.all_tasks

if CFG["task_name"]:
    tasks = [t for t in tasks if t.name == CFG["task_name"]]
if CFG["levels"]:
    allowed = set(CFG["levels"].split(","))
    tasks = [t for t in tasks if str(t.level) in allowed]

print(f"{len(tasks)} tasks selected")

In [ ]:
# ── 8. Main optimisation loop ─────────────────────────────────────────────────
scores_by_task = {}

for task in tasks:
    exp_path = (
        f"out/{CFG['name']}/{timestamp}/{task.category}/{task.name}/{task.level}/"
    )
    task_key = f"{task.name}/{task.level}"

    image_paths = sorted(
        os.path.abspath(p)
        for pat in ["/**/*.png", "/**/*.jpg", "/**/*.jpeg", "/**/*.tif"]
        for p in glob.glob(CFG["dataset_path"] + pat, recursive=True)
    )
    assert image_paths, "No images found — check dataset_path!"
    if CFG["n_images"]:
        image_paths = image_paths[: CFG["n_images"]]

    scores_by_task.setdefault(task_key, {"W": [], "W+": [], "W++": [], "pFID": None})

    # ── LR for this level ─────────────────────────────────────────────────────
    lvl    = task.level
    lr_W   = phase_lr("W",   "LBFGS", lvl, CFG["global_lr_scale"])
    lr_Wp  = phase_lr("W+",  "Adam",  lvl, CFG["global_lr_scale"])
    lr_Wpp = phase_lr("W++", "NGD",   lvl, CFG["global_lr_scale"])

    print(f"\n{'='*72}")
    print(f"  {task_key}")
    print(f"  W(L-BFGS) lr={lr_W:.5f} | W+(Adam) lr={lr_Wp:.5f} | W++(NGD) lr={lr_Wpp:.6f}")

    with directory(exp_path):
        for j, img_path in enumerate(image_paths):
            with directory(f"inversions/{j:04d}"):
                print(f"  img {j:04d}: {os.path.basename(img_path)}")

                ground_truth = open_image(img_path, CFG["resolution"])
                degradation  = task.init_degradation()
                save_image(ground_truth, "ground_truth.png")
                target = degradation.degrade_ground_truth(ground_truth)
                save_image(target, "target.png")

                # ── Phase 1 : W — L-BFGS (adaptive LR) ───────────────────────
                W_var   = WVariable.sample_from(G)
                w_sc    = run_phase(
                    "W", W_var, LBFGSPhase, lr_W,
                    degradation, target, ground_truth,
                    CFG["steps"], CFG["resolution"],
                )
                scores_by_task[task_key]["W"].append(w_sc)

                # ── Phase 2 : W+ — Adam (adaptive LR) ────────────────────────
                Wp_var  = WpVariable.from_W(W_var)
                wp_sc   = run_phase(
                    "W+", Wp_var, torch.optim.Adam, lr_Wp,
                    degradation, target, ground_truth,
                    CFG["steps"], CFG["resolution"],
                )
                scores_by_task[task_key]["W+"].append(wp_sc)

                # ── Phase 3 : W++ — NGD ───────────────────────────────────────
                Wpp_var = WppVariable.from_Wp(Wp_var)
                wpp_sc  = run_phase(
                    "W++", Wpp_var, NGD, lr_Wpp,
                    degradation, target, ground_truth,
                    CFG["steps"], CFG["resolution"],
                )
                scores_by_task[task_key]["W++"].append(wpp_sc)

                print(
                    f"    W  PSNR {w_sc['PSNR']:6.2f} dB  LPIPS {w_sc['LPIPS']:.4f} | "
                    f"W+  PSNR {wp_sc['PSNR']:6.2f} dB  LPIPS {wp_sc['LPIPS']:.4f} | "
                    f"W++ PSNR {wpp_sc['PSNR']:6.2f} dB  LPIPS {wpp_sc['LPIPS']:.4f}"
                )

        # ── pFID ──────────────────────────────────────────────────────────────
        abs_exp    = os.getcwd()
        pred_paths = [
            p for p in [
                os.path.join(abs_exp, "inversions", f"{j:04d}", "pred_W++.png")
                for j in range(len(image_paths))
            ] if os.path.exists(p)
        ]
        if pred_paths:
            pfid = compute_pfid(pred_paths, abs_exp)
            scores_by_task[task_key]["pFID"] = pfid
            print(f"  pFID ({task_key}): {f'{pfid:.2f}' if pfid else 'failed'}")

    print(f"\n--- Running averages for {task_key} ---")
    for ph in ["W", "W+", "W++"]:
        ents = scores_by_task[task_key][ph]
        if ents:
            print(
                f"  {ph:<4} avg PSNR {_mean([s['PSNR']  for s in ents]):6.2f} dB"
                f"  avg LPIPS {_mean([s['LPIPS'] for s in ents]):.4f}"
                f"  (n={len(ents)})"
            )
    pfid = scores_by_task[task_key].get("pFID")
    if pfid:
        print(f"  pFID  {pfid:.2f}")

print("\n=== NOTEBOOK 3 COMPLETE ===")